# **Welcome to the Notebook**

### Task 1 - Set up the project

Installing the needed modules.

In [ ]:
!pip python-dotenv pyspark
%pip install -U openai

ERROR: unknown command "python-dotenv"


Imporint the modules

In [ ]:
from dotenv import load_dotenv
import os
from openai import OpenAI
import pandas as pd
import numpy as np

from pyspark.sql import SparkSession
from pyspark.sql.functions import concat_ws
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, FloatType

from pyspark.ml.feature import VectorAssembler, PCA
from pyspark.ml.clustering import KMeans
import plotly.express as px

Setup the OpenAI API

In [ ]:
load_dotenv(dotenv_path= 'apikey.env.txt')
APIKEY = os.getenv('APIKEY')
print(f"API Key loaded: {APIKEY}") # Added for debugging
client = OpenAI(api_key=APIKEY)


Create a Spark session

In [ ]:
spark = SparkSession.builder.appName("ProductRecommenderSystem").getOrCreate()

Loading the dataset

In [ ]:
file_path ='products_dataset.csv'
df = spark.read.csv(file_path, header=True, inferSchema=True,samplingRatio=1)
df.show()

+----------+--------------------+--------------------+
|product_id|               title|         description|
+----------+--------------------+--------------------+
|        P0|Men's 3X Large Ca...|This heavyweight,...|
|        P1|Turmode 30 ft. RP...|If you need more ...|
|        P2|Large Tapestry Bo...|Polyester cover r...|
|        P3|16-Gauge-Sinks Ve...|It features a rec...|
|        P4|Men's Crazy Horse...|This 9 in. black ...|
|        P5|Mariana 6 ft. Mul...|With robust struc...|
|        P6|5 gal. #650C-2 Po...|BEHR PRO i300 Sem...|
|        P7|7/8 in. x 4-1/2 i...|DEWALT High Perfo...|
|        P8|  Ring Gold Bar Cart|This Ring Bar Car...|
|        P9|Traditional Silve...|This transitional...|
|       P10|15 in. x 59 in. O...|Its easy to add a...|
|       P11|1 qt. #350F-7 Wil...|BEHR PREMIUM PLUS...|
|       P12|Anthracite Cordle...|BlindsAvenue ligh...|
|       P13|SlimGrip 78-Inch ...|Luverne SlimGrip ...|
|       P14|6 in. x 28 in. x ...|Our Rustic Collec...|
|       P1

List of 8 products recently viewed by the user.

In [ ]:
recently_viewed_products = [
    'P316',
    'P333',
    'P1115',
    'P1691',
    'P1082',
    'P397',
    'P1441',
    'P1054',
]

### Task 2 - Prepare the dataset

Combine `title` and `description` Columns

In [ ]:
df = df.withColumn("combined_text", concat_ws(" ", "title", "description"))
df.show()

+----------+--------------------+--------------------+--------------------+
|product_id|               title|         description|       combined_text|
+----------+--------------------+--------------------+--------------------+
|        P0|Men's 3X Large Ca...|This heavyweight,...|Men's 3X Large Ca...|
|        P1|Turmode 30 ft. RP...|If you need more ...|Turmode 30 ft. RP...|
|        P2|Large Tapestry Bo...|Polyester cover r...|Large Tapestry Bo...|
|        P3|16-Gauge-Sinks Ve...|It features a rec...|16-Gauge-Sinks Ve...|
|        P4|Men's Crazy Horse...|This 9 in. black ...|Men's Crazy Horse...|
|        P5|Mariana 6 ft. Mul...|With robust struc...|Mariana 6 ft. Mul...|
|        P6|5 gal. #650C-2 Po...|BEHR PRO i300 Sem...|5 gal. #650C-2 Po...|
|        P7|7/8 in. x 4-1/2 i...|DEWALT High Perfo...|7/8 in. x 4-1/2 i...|
|        P8|  Ring Gold Bar Cart|This Ring Bar Car...|Ring Gold Bar Car...|
|        P9|Traditional Silve...|This transitional...|Traditional Silve...|
|       P10|

get the combined_text column and convert it into a list

In [ ]:
list_combined_text = df.select("combined_text").rdd.flatMap(lambda x: x).collect()
print(list_combined_text[:4])

["Men's 3X Large Carbon Heather Cotton/Polyester Rain Defender Paxton Heavyweight Hooded Zip-Front Sweatshirt This heavyweight, water-repellent hooded sweatshirt has a zip front for fast layering. ORIGINAL FIT. 13 oz., 75% cotton/25% polyester blend with Rain Defender durable water repellent. Attached, jersey-lined three-piece hood with drawcord closure. Antique-finish brass front zipper. Two front hand-warmer pockets have a hidden security pocket inside. Stretchable, spandex-reinforced rib-knit cuffs and waistband. Locker loop facilitates hanging.", "Turmode 30 ft. RP TNC Female to RP TNC Male Adapter Cable If you need more length between your existing wireless device and Hi-Gain Antenna, this is the product for you. It's compatible with most Wi-Fi Antennas, so it is easy for you to extend your wireless network. Just replace your existing cable that runs between your wireless device and Antenna and you're ready to use your network with extended range.", 'Large Tapestry Bolster Bed Pol

Use OpenAI text embedding model to create the vector embeddings.

In [ ]:
response = client.embeddings.create(
    input=list_combined_text,
    model="text-embedding-3-small",
    dimensions= 512
)
embedding_vectors = [d.embedding for d in response.data]
embedding_vectors[:2]

[[0.04266357421875,
  0.0208740234375,
  -0.01364898681640625,
  -0.002056121826171875,
  0.0031986236572265625,
  -0.037261962890625,
  0.0272216796875,
  0.07830810546875,
  0.05499267578125,
  -0.06280517578125,
  0.04425048828125,
  0.048248291015625,
  -0.06781005859375,
  0.02520751953125,
  0.0225677490234375,
  0.06396484375,
  0.10101318359375,
  -0.0307769775390625,
  -0.0889892578125,
  0.07061767578125,
  -0.058013916015625,
  0.06719970703125,
  -0.034759521484375,
  -0.07373046875,
  0.034454345703125,
  0.043243408203125,
  -0.052520751953125,
  0.0249786376953125,
  0.051055908203125,
  -0.0172119140625,
  -0.0031528472900390625,
  -0.0221099853515625,
  0.01910400390625,
  -0.0245513916015625,
  0.044921875,
  -0.066650390625,
  -0.02105712890625,
  0.10443115234375,
  -0.038665771484375,
  0.025634765625,
  0.02349853515625,
  -0.052764892578125,
  0.0181732177734375,
  -0.06268310546875,
  0.032623291015625,
  0.03680419921875,
  0.0259552001953125,
  0.0240631103515

Let't put the embedding vectors into our original dataframe

Convert embedding vectors list into a Pyspark DataFrame

In [ ]:
features_column_names = [f"embedding_{i}" for i in range(len(embedding_vectors[0]))]
embeddings_df = spark.createDataFrame(embedding_vectors, schema=features_column_names)
embeddings_df.show()

+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+------------------+--------------------+--------------------+---------------

Add unique `row_id` to each row in the pysaprk dataframe

In [ ]:
embeddings_df = embeddings_df.repartition(1).withColumn("row_id", F.monotonically_increasing_id())
embeddings_df.show()

+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+------------------+--------------------+--------------------+---------------

Add unique `row_id` to each row in our main pyspark dataframe `df`

In [ ]:
df = df.repartition(1).withColumn("row_id", F.monotonically_increasing_id())
df.show()

+----------+--------------------+--------------------+--------------------+------+
|product_id|               title|         description|       combined_text|row_id|
+----------+--------------------+--------------------+--------------------+------+
|        P0|Men's 3X Large Ca...|This heavyweight,...|Men's 3X Large Ca...|     0|
|        P1|Turmode 30 ft. RP...|If you need more ...|Turmode 30 ft. RP...|     1|
|        P2|Large Tapestry Bo...|Polyester cover r...|Large Tapestry Bo...|     2|
|        P3|16-Gauge-Sinks Ve...|It features a rec...|16-Gauge-Sinks Ve...|     3|
|        P4|Men's Crazy Horse...|This 9 in. black ...|Men's Crazy Horse...|     4|
|        P5|Mariana 6 ft. Mul...|With robust struc...|Mariana 6 ft. Mul...|     5|
|        P6|5 gal. #650C-2 Po...|BEHR PRO i300 Sem...|5 gal. #650C-2 Po...|     6|
|        P7|7/8 in. x 4-1/2 i...|DEWALT High Perfo...|7/8 in. x 4-1/2 i...|     7|
|        P8|  Ring Gold Bar Cart|This Ring Bar Car...|Ring Gold Bar Car...|     8|
|   

Let's join the two dataframes

In [ ]:
df = df.join(embeddings_df, on="row_id").drop("row_id")
df.show()

+----------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--

### Task 3 - Cluster products using K-means

Assemble the 512 Embedding Columns into a Single 'features' Column

In [ ]:
assembler = VectorAssembler(inputCols=features_column_names, outputCol="features")
data = assembler.transform(df)
data = data.select(['product_id','title','features'])
data.show()

+----------+--------------------+--------------------+
|product_id|               title|            features|
+----------+--------------------+--------------------+
|        P0|Men's 3X Large Ca...|[0.04266357421875...|
|        P1|Turmode 30 ft. RP...|[0.04409790039062...|
|        P2|Large Tapestry Bo...|[0.0423583984375,...|
|        P3|16-Gauge-Sinks Ve...|[-0.0497436523437...|
|        P4|Men's Crazy Horse...|[0.02626037597656...|
|        P5|Mariana 6 ft. Mul...|[0.060546875,0.04...|
|        P6|5 gal. #650C-2 Po...|[0.00754165649414...|
|        P7|7/8 in. x 4-1/2 i...|[-0.0217437744140...|
|        P8|  Ring Gold Bar Cart|[-4.4417381286621...|
|        P9|Traditional Silve...|[0.03915405273437...|
|       P10|15 in. x 59 in. O...|[6.26564025878906...|
|       P11|1 qt. #350F-7 Wil...|[0.00136566162109...|
|       P12|Anthracite Cordle...|[-0.0231170654296...|
|       P13|SlimGrip 78-Inch ...|[0.02252197265625...|
|       P14|6 in. x 28 in. x ...|[0.01872253417968...|
|       P1

Apply K-Means Clustering with 5 Clusters on the `features` Column

In [ ]:
kmeans = KMeans(k=5, featuresCol="features", predictionCol="cluster")
model = kmeans.fit(data)
cluster_data = model.transform(data)
cluster_data.show()

+----------+--------------------+--------------------+-------+
|product_id|               title|            features|cluster|
+----------+--------------------+--------------------+-------+
|        P0|Men's 3X Large Ca...|[0.04266357421875...|      2|
|        P1|Turmode 30 ft. RP...|[0.04409790039062...|      2|
|        P2|Large Tapestry Bo...|[0.0423583984375,...|      4|
|        P3|16-Gauge-Sinks Ve...|[-0.0497436523437...|      3|
|        P4|Men's Crazy Horse...|[0.02626037597656...|      2|
|        P5|Mariana 6 ft. Mul...|[0.060546875,0.04...|      3|
|        P6|5 gal. #650C-2 Po...|[0.00754165649414...|      1|
|        P7|7/8 in. x 4-1/2 i...|[-0.0217437744140...|      2|
|        P8|  Ring Gold Bar Cart|[-4.4417381286621...|      3|
|        P9|Traditional Silve...|[0.03915405273437...|      3|
|       P10|15 in. x 59 in. O...|[6.26564025878906...|      3|
|       P11|1 qt. #350F-7 Wil...|[0.00136566162109...|      1|
|       P12|Anthracite Cordle...|[-0.0231170654296...| 

### Task 4 - Visualize the clusters

Let's reduce the dimensionality of our features for visualization purpose

`512 dimensions => 2 dimensions`

In [ ]:
pca = PCA(k=2, inputCol="features", outputCol="pca_features")
pca_model = pca.fit(cluster_data )
pca_results = pca_model.transform(cluster_data )
pca_results.show()

+----------+--------------------+--------------------+-------+--------------------+
|product_id|               title|            features|cluster|        pca_features|
+----------+--------------------+--------------------+-------+--------------------+
|        P0|Men's 3X Large Ca...|[0.04266357421875...|      2|[0.18857446687517...|
|        P1|Turmode 30 ft. RP...|[0.04409790039062...|      2|[-0.1739726020281...|
|        P2|Large Tapestry Bo...|[0.0423583984375,...|      4|[-0.0202065381698...|
|        P3|16-Gauge-Sinks Ve...|[-0.0497436523437...|      3|[0.00733770024539...|
|        P4|Men's Crazy Horse...|[0.02626037597656...|      2|[-0.0239812804463...|
|        P5|Mariana 6 ft. Mul...|[0.060546875,0.04...|      3|[-0.0014707727159...|
|        P6|5 gal. #650C-2 Po...|[0.00754165649414...|      1|[0.66985660000214...|
|        P7|7/8 in. x 4-1/2 i...|[-0.0217437744140...|      2|[0.07966090622713...|
|        P8|  Ring Gold Bar Cart|[-4.4417381286621...|      3|[-0.1048041627

In [ ]:
pca_df = pca_results.select("product_id", "pca_features","cluster").toPandas()
pca_df.head()
pca_df["x"] = pca_df["pca_features"].apply(lambda x: x[0])
pca_df["y"] = pca_df["pca_features"].apply(lambda x: x[1])


Let's plot the Clusters

In [ ]:
def plot_clusters(pca_df, num_clusters=5):
    """
    Plots a 2D visualization of clusters using Plotly Express.

    Parameters:
    - pca_df (DataFrame): A Pandas DataFrame containing columns 'x', 'y', and 'cluster'.
      'x' and 'y' are the 2D PCA components, and 'cluster' indicates the cluster label.
    - num_clusters (int): The number of unique clusters to display.
    - recently_viewed_df (DataFrame, optional): DataFrame with 'x' and 'y' coordinates for recently viewed products.

    This function creates an interactive scatter plot where each point is colored according to its cluster.
    Recently viewed products are marked as black crosses if provided.

    Returns:
    - fig (Figure): The Plotly figure object for the plot.
    """

    # Create the base cluster plot
    fig = px.scatter(
        pca_df,
        x='x',
        y='y',
        opacity=0.6,
        size_max=4,
        color= pca_df.cluster.astype(str),
        title='2D Visualization of Clusters with Recently Viewed Products',
        labels={'x': 'PCA Component 1', 'y': 'PCA Component 2'},
        category_orders={'cluster': list(range(num_clusters))},
        # show the product id in the tooltip
        hover_data={'product_id': True}

    )

    # Update layout to add legend title and adjust plot settings
    fig.update_layout(legend_title_text='Clusters', legend=dict(x=1, y=1), width=600, height=500)

    return fig

fig = plot_clusters(pca_df)
fig.show()

### Task 5 - Highlight recently viewed products

In [ ]:
print("The user has recently viewed the following products: ", recently_viewed_products)

The user has recently viewed the following products:  ['P316', 'P333', 'P1115', 'P1691', 'P1082', 'P397', 'P1441', 'P1054']


Let's have a look at the records in our `clustered_data` dataframe related to the recently viewed products.

In [ ]:
filtered_data = cluster_data.where(F.col("product_id").isin(recently_viewed_products))
unique_clusters = filtered_data.select("cluster").distinct().rdd.flatMap(lambda x: x).collect()
unique_clusters

[4, 1]

### Task 6 - Recommend products based on recently viewed products

Let's have a look at the recently viewed products titles

In [ ]:
filtered_data.select("title").rdd.flatMap(lambda x: x).collect()

["Mystic Fitz Roy Beige 9' 0 x 12' 0 Area Rug",
 'Florida Shag Beige/Multi 3 ft. x 5 ft. Floral Area Rug',
 '1 gal. #M250-3 Apple Turnover Extra Durable Flat Interior Paint & Primer',
 '1 gal. #HDPG60 Misty Emerald Lake Flat Interior Paint and Primer',
 '1 qt. #S220-7 Molasses Extra Durable Flat Interior Paint & Primer',
 'Modern Gray/Multi 9 ft. x 12 ft. Vibrant Abstract Polyester Area Rug',
 '1 qt. #PPU6-06 Honey Locust Eggshell Enamel Low Odor Interior Paint & Primer',
 'Genet Rust/Red-Brown 8 ft. x 11 ft. Abstract Wool Area Rug']

Let's see the distinct clusters of the recenetly viewed products.

In [ ]:
print(unique_clusters)

[4, 1]


Let's find the possible products for the recommendation.

In [ ]:
possible_recommendations = cluster_data.filter(cluster_data['cluster'].isin(unique_clusters)).filter(~cluster_data['product_id'].isin(recently_viewed_products))

Let's perform a groupby and generate a list of product IDs that can be recommended for each of the clusters.

In [ ]:
recommendations = possible_recommendations.groupby("cluster").agg(
    F.collect_list("product_id").alias("recommendations")
)

recommendations_df = recommendations.toPandas()

recommendations_df["random_recommendations"] = (
    recommendations_df["recommendations"]
    .apply(lambda x: np.random.choice(x, 5, replace=False).tolist())
)

recommendations_df.head()

,cluster,recommendations,random_recommendations
0,4,"[P2, P21, P52, P71, P87, P101, P108, P119, P12...","[P967, P1502, P101, P1001, P1168]"
1,1,"[P6, P11, P16, P18, P24, P26, P30, P33, P40, P...","[P81, P418, P182, P159, P729]"


In [ ]:
# write a python function to display the recommendations
def display_recommendations(row):
  # find the title of the product in df
  product_ids = row['random_recommendations']
  cluster = row.cluster

  titles = data. \
          filter(data["product_id"]. \
          isin(product_ids)).select("title").collect()

  print("\n")
  print("Recommendations for Cluster:", cluster)
  for title in titles:
    print(title[0])

recommendations_df.apply(display_recommendations, axis=1)



Recommendations for Cluster: 4
Pueblo Multi-Colored 3 ft. x 5 ft. Native American Area Rug
Blossom Ivory/Gray 2 ft. x 8 ft. Geometric Aztec Runner Rug
Galena Orange/Brown 9 ft. x 12 ft. Medallion Rectangle Area Rug
Chelsea Red 3 ft. x 14 ft. Border Runner Rug
Drey Ombre Shag Sky Blue 4 ft. x 6 ft. Area Rug


Recommendations for Cluster: 1
1 gal. Home Decorators Collection #HDC-FL14-8 Deer Trail Semi-Gloss Enamel Exterior Paint & Primer
1 qt. #PPU7-03A Sofisticata Hi-Gloss Enamel Interior/Exterior Paint
8 oz. #M530-3 Perennial Blue Satin Enamel Interior/Exterior Paint & Primer Color Sample
1 qt. Black Cherry Water-Based Satin Metallic Interior Paint
1 qt. #T12-20 First Peach Eggshell Enamel Interior Stain -Blocking Paint & Primer


,0
0,None
1,None
